In [1]:
import sys, os
sys.path.append("/pscratch/sd/a/awikner/PanguWeather-ens/v2.0")
from ensemble_inference import Stepper

/global/homes/a/awikner/.conda/envs/py311_pip_sfno/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
from ruamel.yaml.comments import CommentedMap as ruamelDict
from ruamel.yaml import YAML
from utils.YParams import YParams
import cftime
params = YParams("/pscratch/sd/a/awikner/PanguWeather-ens/v2.0/config/PANGU_PLASIM_H5_PERLMUTTER_0515_enstest.yaml", "PLASIM")
#params['epsilon_factor'] = args.epsilon_factor
params['run_iter'] = 1
if hasattr(params, 'diagnostic_variables'):
    if len(params.diagnostic_variables) > 0:
        params['has_diagnostic'] = True
    else:
        params['has_diagnostic'] = False
else:
    params['has_diagnostic'] = False
print(f'Has diagnostic: {params.has_diagnostic}')

expDir = os.path.join(params.exp_dir, "PLASIM", "0515")
if not os.path.isdir(expDir):
    os.makedirs(expDir)
    os.makedirs(os.path.join(expDir, 'training_checkpoints/'))

params['experiment_dir'] = os.path.abspath(expDir)
ckpt_path_globstr = 'training_checkpoints/ckpt_*.tar'
best_ckpt_path = 'training_checkpoints/best_ckpt.tar'
params['checkpoint_path_globstr'] = os.path.join(expDir, ckpt_path_globstr)
params['best_checkpoint_path'] = os.path.join(expDir, best_ckpt_path)

params['init_datetime'] = cftime.datetime(params.val_year_start, 1, 1, 0, has_year_zero = params.has_year_zero, calendar = 'proleptic_gregorian')
params['init_datetime'] = cftime.DatetimeProlepticGregorian(params.init_datetime.year,
                                                            params.init_datetime.month,
                                                            params.init_datetime.day,
                                                            hour = params.init_datetime.hour,
                                                            has_year_zero = params.has_year_zero)
params["world_size"] = 0
params['world_rank'] = 0
params['local_rank'] = 0
params['init_nc_filepaths'] = ["/pscratch/sd/a/awikner/PLASIM/data/test_data/copy_0/test_data_postproc.nc"]
params['enable_amp'] = False

Has diagnostic: True


In [3]:
stepper = Stepper(params, 0)

2025-03-23 14:53:00,716 - root - INFO - rank 0, begin data loader init


Loading boundary data: 100%|██████████| 120/120 [00:00<00:00, 140.47it/s]
2025-03-23 14:53:02,013 - root - INFO - Ensemble Mode. Ensemble size = {params.num_ensemble_members}

2025-03-23 14:53:02,014 - root - INFO - rank 0, data loader initialized
/global/homes/a/awikner/.conda/envs/py311_pip_sfno/lib/python3.11/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Embedding Dimensions are 192
Num diagnostic vars: 1
Pad sizes in 3D patch embedding
(0, 0, 0, 0, 1, 0)
Patch Size
[2, 2, 2]
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
Embed Dim:
192
Input resolution:
(8, 32, 64)
depth
2
num_heads
6
drop_path:
[0.0, 0.028571428571428574]
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0)


2025-03-23 14:53:02,581 - root - INFO - Number of trainable model parameters: 28791944


START EPOCH: 99


In [4]:
stepper.model

PanguModel_Plasim(
  (patchembed2d): PatchEmbed2D(
    (pad): ZeroPad2d((0, 0, 0, 0))
    (proj): Conv2d(8, 192, kernel_size=(2, 2), stride=(2, 2))
  )
  (patchembed3d): PatchEmbed3D(
    (pad): ZeroPad3d((0, 0, 0, 0, 1, 0))
    (proj): Conv3d(5, 192, kernel_size=(2, 2, 2), stride=(2, 2, 2))
  )
  (layer1): EarthSpecificLayer(
    (blocks): ModuleList(
      (0): EarthSpecificBlock(
        (norm1): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
        (pad): ZeroPad3d((0, 0, 0, 0, 0, 0))
        (attn): EarthAttention3D(
          (qkv): Linear(in_features=192, out_features=576, bias=True)
          (proj): Linear(in_features=192, out_features=192, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj_drop): Dropout(p=0.0, inplace=False)
          (softmax): Softmax(dim=-1)
        )
        (drop_path): Identity()
        (mlp): Mlp(
          (fc1): Linear(in_features=192, out_fe

In [5]:
import torch
torch._dynamo.config.verbose = True
#torch._dynamo.config.suppress_errors = False
device = torch.cuda.current_device()
example_inputs = (torch.randn(1, 2, 64, 128, dtype = torch.float32, device = device),
                  torch.randn(1, 3, 64, 128, dtype = torch.float32, device = device),
                  torch.randn(1, 3, 64, 128, dtype = torch.float32, device = device),
                  torch.randn(1, 5, 13, 64, 128, dtype = torch.float32, device = device))
compiled_model = torch.compile(stepper.model, mode = "max-autotune")
with torch.inference_mode():
    out = compiled_model(*example_inputs)

/global/homes/a/awikner/.conda/envs/py311_pip_sfno/lib/python3.11/site-packages/torch/_dynamo/variables/functions.py:679: UserWarning: Graph break due to unsupported builtin None.Tensor.view. This function is either a Python builtin (e.g. _warnings.warn) or a third-party C/C++ Python extension (perhaps created with pybind). If it is a Python builtin, please file an issue on GitHub so the PyTorch team can add support for it and see the next case for a workaround. If it is a third-party C/C++ Python extension, please either wrap it into a PyTorch-understood custom operator (see https://pytorch.org/tutorials/advanced/custom_ops_landing_page.html for more details) or, if it is traceable, use torch.compiler.allow_in_graph.
  torch._dynamo.utils.warn_once(msg)
W0323 14:53:16.474000 2190999 /global/u2/a/awikner/.conda/envs/py311_pip_sfno/lib/python3.11/site-packages/torch/_dynamo/convert_frame.py:906] [10/8] torch._dynamo hit config.cache_size_limit (8)
W0323 14:53:16.474000 2190999 /global/u

In [7]:
import time
from tqdm.notebook import tqdm
N = 100
example_inputs = (torch.randn(N, 2, 64, 128, dtype = torch.float32, device = device),
                  torch.randn(N, 3, 64, 128, dtype = torch.float32, device = device),
                  torch.randn(N, 3, 64, 128, dtype = torch.float32, device = device),
                  torch.randn(N, 5, 13, 64, 128, dtype = torch.float32, device = device))
model_time = 0
compiled_time = 0
with torch.inference_mode():
    for i in tqdm(range(N)):
        model_start = time.time()
        out = stepper.model(example_inputs[0][i].unsqueeze(0),
                            example_inputs[1][i].unsqueeze(0),
                            example_inputs[2][i].unsqueeze(0),
                            example_inputs[3][i].unsqueeze(0))
        model_time += time.time() - model_start
        compiled_start = time.time()
        out = compiled_model(example_inputs[0][i].unsqueeze(0),
                            example_inputs[1][i].unsqueeze(0),
                            example_inputs[2][i].unsqueeze(0),
                            example_inputs[3][i].unsqueeze(0))
        compiled_time += time.time() - compiled_start
print(model_time)
print(compiled_time)

  0%|          | 0/100 [00:00<?, ?it/s]

2.8211071491241455
3.1742420196533203


In [6]:
onnx_program.optimize()

2025-03-23 13:39:07,739 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_61 due to large size 6291456.


2025-03-23 13:39:07,796 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_148 due to large size 6291456.
2025-03-23 13:39:07,858 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_242 due to large size 3145728.
2025-03-23 13:39:08,175 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_330 due to large size 3145728.
2025-03-23 13:39:08,231 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_409 due to large size 3145728.
2025-03-23 13:39:08,292 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_492 due to large size 3145728.
2025-03-23 13:39:08,347 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_570 due to large size 3145728.
2025-03-23 13:39:08,409 - onnxscript.optimizer._constant_folding - INFO - Skip storing constant folded nvalue val_653 due to large

Applied 50 of general pattern rewrite rules.


2025-03-23 13:39:09,592 - onnxscript.rewriter.collapse_slices - INFO - The value 'end' is less than the shape of the specified axis.
2025-03-23 13:39:09,592 - onnxscript.rewriter.collapse_slices - INFO - The value 'start' is not 0.
2025-03-23 13:39:09,592 - onnxscript.rewriter.collapse_slices - INFO - The value 'start' is not 0.
2025-03-23 13:39:09,620 - onnxscript.optimizer._remove_unused - INFO - Removed 50 unused nodes
2025-03-23 13:39:09,623 - onnxscript.optimizer._remove_unused_function - INFO - No unused functions to remove
2025-03-23 13:39:09,646 - onnxscript.optimizer._remove_unused - INFO - Removed 0 unused nodes


In [7]:
onnx_program.save('.'.join(params['best_checkpoint_path'].split('.')[:-1]) + '.onnx')

In [8]:
onnx_program(example_inputs)

(tensor([[[[ 0.3969,  0.1875, -0.3068,  ..., -0.0210,  0.0161,  0.0891],
           [ 0.4657, -0.1348, -0.3388,  ..., -0.4651,  0.2650,  0.7405],
           [ 0.2302, -0.4942,  0.1075,  ..., -0.8436,  0.1061,  1.0310],
           ...,
           [-0.0491, -0.0110, -0.1410,  ..., -0.3194,  0.0387,  0.3692],
           [-0.1989, -0.0376, -0.3568,  ..., -0.1529, -0.4134, -0.1093],
           [-0.0479,  0.1807,  0.0227,  ..., -0.2984, -0.4603, -0.2235]],
 
          [[ 0.2616,  0.1915,  0.0626,  ...,  0.0682,  0.0120,  0.0615],
           [ 0.2863,  0.1699,  0.0541,  ...,  0.0403,  0.0635,  0.2841],
           [ 0.2488,  0.0431,  0.0354,  ..., -0.0604,  0.1013,  0.3434],
           ...,
           [ 0.1716,  0.0881,  0.0753,  ...,  0.1314,  0.1855,  0.3346],
           [ 0.0224,  0.1224,  0.1366,  ..., -0.0465, -0.1127, -0.0035],
           [-0.1085, -0.0257,  0.0194,  ..., -0.1780, -0.3128, -0.2041]]]]),
 tensor([[[[[-3.6312e-02, -7.5210e-02, -1.6554e-01,  ..., -8.6510e-02,
             -

In [4]:
import numpy as np
x = np.load('/pscratch/sd/a/awikner/PLASIM/data/test_data/copy_3/PanguPlasim_0_0000-0100_A_France.npy')

In [5]:
import h5py, os, sys, glob
import numpy as np
from natsort import natsorted
from tqdm.notebook import tqdm

from utils.data_loader_multifiles import get_data_given_path
data_dir = '/eagle/MDClimSim/awikner/PLASIM/data/h5/plev_data'

In [6]:
year_files = natsorted(glob.glob(os.path.join(data_dir, '11_*.h5')))
tisr_data = np.zeros((len(year_files), 64, 128), dtype = np.float32)
tas_data = np.zeros((len(year_files), 64, 128), dtype = np.float32)
for i, file in tqdm(enumerate(year_files)):
    data_in = get_data_given_path(file, ['rsdt', 'tas'])
    tisr_data[i] = data_in[0]
    tas_data[i] = data_in[1]

0it [00:00, ?it/s]

In [ ]:
import xarray as xr
from datetime import timedelta
ds = xr.open_dataset('/eagle/MDClimSim/awikner/PLASIM/data/train_val_test_data_res/data_11.nc')
ds_2 = ds.copy(deep = True)
ds_2['time'] = ds_2['time'] - timedelta(hours = 18)
ds['rsdt'] = xr.DataArray(tisr_data, dims = ("time", "lat", "lon"))
ds['tas_2'] = xr.DataArray(tas_data, dims = ("time", "lat", "lon"))
ds_2['rsdt'] = xr.DataArray(tisr_data, dims = ("time", "lat", "lon"))
ds_2['tas_2'] = xr.DataArray(tas_data, dims = ("time", "lat", "lon"))

/soft/applications/conda/2024-04-29/mconda3/lib/python3.11/site-packages/xarray/coding/times.py:995: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates out of range
  dtype = _decode_cf_datetime_dtype(data, units, calendar, self.use_cftime)
/soft/applications/conda/2024-04-29/mconda3/lib/python3.11/site-packages/xarray/core/indexing.py:630: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates out of range
  array = array.get_duck_array()
/soft/applications/conda/2024-04-29/mconda3/lib/python3.11/site-packages/xarray/core/indexing.py:500: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates out of range
  return np.asarray(self.get_duck_array(), dtype=dtype)


In [34]:
print(ds.tas.isel(time = 4, lat = 16, lon = 1).values)
print(tas_data[7, 16, 1])

277.74384
277.74384


In [25]:
print(ds.tas_2.isel(time = slice(4), lat = 16, lon = 1).values)

[278.06326 276.971   277.74356 277.2512 ]


In [27]:
import matplotlib.pyplot as plt
ds_jja = ds.where(ds.time.dt.month.isin([6, 7, 8]), drop=True).isel(lat = 16, lon = 1)
#print(ds_jja.rsdt.groupby(ds_jja.time.dt.hour).mean('time'))
print(ds_jja.tas.groupby(ds_jja.time.dt.hour).mean('time'))
#print(ds_jja.tas_2.groupby(ds_jja.time.dt.hour).mean('time'))
ds_jja = ds_2.where(ds_2.time.dt.month.isin([6, 7, 8]), drop=True).isel(lat = 16, lon = 1)
print(ds_jja.rsdt.groupby(ds_jja.time.dt.hour).mean('time'))
#print(ds_jja.tas.groupby(ds_jja.time.dt.hour).mean('time'))
print(ds_jja.tas_2.groupby(ds_jja.time.dt.hour).mean('time'))
#plt.show()

<xarray.DataArray 'tas' (hour: 4)> Size: 16B
array([301.29886, 298.19434, 300.5584 , 304.754  ], dtype=float32)
Coordinates:
    lon      float64 8B 2.812
    lat      float64 8B 43.25
  * hour     (hour) int64 32B 0 6 12 18
Attributes:
    standard_name:  air_temperature_2m
    long_name:      air_temperature_2m
    units:          K
    code:           167
    CDI_grid_type:  gaussian
<xarray.DataArray 'rsdt' (hour: 4)> Size: 16B
array([ 19.512693,  55.559   , 940.66144 , 832.8492  ], dtype=float32)
Coordinates:
    lon      float64 8B 2.812
    lat      float64 8B 43.25
  * hour     (hour) int64 32B 0 6 12 18
<xarray.DataArray 'tas_2' (hour: 4)> Size: 16B
array([301.29886, 298.19434, 300.5584 , 304.754  ], dtype=float32)
Coordinates:
    lon      float64 8B 2.812
    lat      float64 8B 43.25
  * hour     (hour) int64 32B 0 6 12 18


In [12]:
from datetime import timedelta
ds_2_tas = ds.tas
ds_2_tas['time'] = ds_2_tas['time'] + timedelta(hours = 18)
ds_2_jja = ds_2_tas.where(ds_2_tas.time.dt.month.isin([6, 7, 8]), drop=True).isel(lat = 16, lon = 1)
print(ds_2_jja.groupby(ds_2_jja.time.dt.hour).mean('time'))

<xarray.DataArray 'tas' (hour: 4)> Size: 16B
array([298.1005 , 300.46655, 304.65714, 301.29886], dtype=float32)
Coordinates:
    lon      float64 8B 2.812
    lat      float64 8B 43.25
  * hour     (hour) int64 32B 0 6 12 18
Attributes:
    standard_name:  air_temperature_2m
    long_name:      air_temperature_2m
    units:          K
    code:           167
    CDI_grid_type:  gaussian


In [18]:
ds_era5 = xr.open_dataset('/eagle/MDClimSim/tungnd/data/wb2/6h_721_1440_with_poles/2m_temperature/1995.nc')
ds_jja = ds_era5.where(ds_era5.time.dt.month.isin([6, 7, 8]), drop=True).sel(latitude = 43.25, longitude = 2.75, method = 'nearest')
print(ds_jja['2m_temperature'].groupby(ds_jja.time.dt.hour).mean('time'))

<xarray.DataArray '2m_temperature' (hour: 4)> Size: 16B
array([291.50418, 291.46982, 298.7399 , 297.24918], dtype=float32)
Coordinates:
    latitude   float32 4B 43.25
    longitude  float32 4B 2.75
  * hour       (hour) int64 32B 0 6 12 18
Attributes:
    long_name:   2 metre temperature
    short_name:  t2m
    units:       K


In [17]:
ds_rsdt = xr.open_dataset('/eagle/MDClimSim/awikner/PLASIM/data/train_val_test_data_res/boundary_vars/rsdt_leap.nc')
print(ds_rsdt.rsdt.isel(lat = 16, lon = 1).groupby(ds_rsdt.time.dt.hour).mean())

/soft/applications/conda/2024-04-29/mconda3/lib/python3.11/site-packages/xarray/coding/times.py:995: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates out of range
  dtype = _decode_cf_datetime_dtype(data, units, calendar, self.use_cftime)
/soft/applications/conda/2024-04-29/mconda3/lib/python3.11/site-packages/xarray/core/indexing.py:630: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates out of range
  array = array.get_duck_array()
/soft/applications/conda/2024-04-29/mconda3/lib/python3.11/site-packages/xarray/core/indexing.py:500: SerializationWarning: Unable to decode time axis into full numpy.datetime64 objects, continuing using cftime.datetime objects instead, reason: dates out of range
  return np.asarray(self.get_duck_array(), dtype=dtype)


<xarray.DataArray 'rsdt' (hour: 4)> Size: 16B
array([  6.3005986,  20.567255 , 667.6452   , 569.3823   ], dtype=float32)
Coordinates:
    lon      float64 8B 2.812
    lat      float64 8B 43.25
  * hour     (hour) int64 32B 0 6 12 18
